# Movie Dialogue Relationship Extraction and Knowledge Graph

This notebook builds a character relationship knowledge graph from the Cornell Movie Dialogs Corpus, then uses the extracted relationship context to translate dialogue line by line into Traditional Chinese.

The implementation is kept in `movie_kg_pipeline.py` so the notebook and command-line workflow share the same code.

## 1. Setup

Install dependencies if needed, then set `GEMINI_API_KEY` in your environment or in a local `.env` file. The default model is `gemini-2.5-flash-lite`, chosen for high free-tier request capacity among Gemini 2.5 text models.

In [ ]:
# Uncomment if packages are missing.
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json
import pandas as pd

from movie_kg_pipeline import (
    DEFAULT_MODEL,
    build_character_canonical_map,
    build_movie_dialogue,
    build_nodes_table,
    build_turn_objects,
    chunk_dialogue_turns,
    extract_chunk_with_gemini,
    find_movie_by_title,
    GeminiJsonClient,
    load_cornell_dataset,
    merge_relationships,
    mock_extract_from_chunk,
    relationship_context_by_speaker,
    translate_turn_batch_with_gemini,
    mock_translate_turns,
    batched,
    save_json,
)

## 2. Configuration

Set `RUN_LLM = True` when you are ready to call Gemini. Keep small limits while testing to avoid spending quota too quickly.

In [ ]:
DATA_DIR = Path("cornell_movie_dialogs_corpus")
OUTPUT_DIR = Path("outputs_cornell_gemini")
CHUNK_OUTPUT_DIR = OUTPUT_DIR / "chunk_results"
TRANSLATION_OUTPUT_DIR = OUTPUT_DIR / "translations"

MOVIE_TITLE_QUERY = "Titanic"
GEMINI_MODEL = DEFAULT_MODEL
RUN_LLM = False
RUN_TRANSLATION = False

MAX_CHUNKS_TO_RUN = 2
MAX_TRANSLATION_TURNS = 20
TRANSLATION_BATCH_SIZE = 20

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRANSLATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3. Load the Cornell Dataset

In [ ]:
df_lines, df_convs, df_titles, df_chars = load_cornell_dataset(DATA_DIR)

print("Lines:", len(df_lines))
print("Conversations:", len(df_convs))
print("Movies:", len(df_titles))
print("Characters:", len(df_chars))

df_titles.head()

## 4. Select a Movie and Reconstruct Dialogue Turns

In [ ]:
movie_matches = find_movie_by_title(df_titles, MOVIE_TITLE_QUERY)
if movie_matches.empty:
    raise ValueError(f"No movie title matched: {MOVIE_TITLE_QUERY}")

selected_movie = movie_matches.iloc[0]
SELECTED_MOVIE_ID = selected_movie["movie_id"]
SELECTED_MOVIE_TITLE = selected_movie["title"]

movie_turns_df = build_movie_dialogue(SELECTED_MOVIE_ID, df_lines, df_convs)
print("Selected movie:", SELECTED_MOVIE_ID, SELECTED_MOVIE_TITLE)
print("Dialogue turns:", len(movie_turns_df))

movie_turns_df.head(10)

## 5. Inspect Speaker Coverage

The Cornell corpus already assigns a speaker to each dialogue line. We use `speaker_name` as the first-layer character identifier and later merge aliases into canonical graph nodes.

In [ ]:
speaker_counts = movie_turns_df["speaker_name"].value_counts().reset_index()
speaker_counts.columns = ["speaker_name", "turn_count"]
speaker_counts.head(20)

## 6. Chunk Dialogue for LLM Extraction

In [ ]:
turns = build_turn_objects(movie_turns_df)
chunks = chunk_dialogue_turns(
    turns,
    movie_id=SELECTED_MOVIE_ID,
    movie_title=SELECTED_MOVIE_TITLE,
    max_turns_per_chunk=40,
    overlap_turns=8,
    max_approx_chars_per_chunk=6000,
)

print("Chunks:", len(chunks))
print(chunks[0].chunk_id, chunks[0].start_turn, chunks[0].end_turn)
print(chunks[0].to_prompt_text()[:1500])

## 7. Extract Characters and Relationships

When `RUN_LLM` is `False`, this cell uses a mock extractor so the rest of the notebook can be tested without API calls. Set `RUN_LLM = True` to call Gemini.

In [ ]:
client = GeminiJsonClient(api_key="", model=GEMINI_MODEL) if RUN_LLM else None
selected_chunks = chunks[:MAX_CHUNKS_TO_RUN] if MAX_CHUNKS_TO_RUN is not None else chunks
chunk_results = []

for index, chunk in enumerate(selected_chunks, start=1):
    print(f"[{index}/{len(selected_chunks)}] {chunk.chunk_id}")
    if RUN_LLM:
        result = extract_chunk_with_gemini(chunk, client=client)
    else:
        result = mock_extract_from_chunk(chunk)
    payload = result.model_dump()
    save_json(CHUNK_OUTPUT_DIR / f"{chunk.chunk_id}.json", payload)
    chunk_results.append(payload)

print(json.dumps(chunk_results[0], ensure_ascii=False, indent=2)[:2500])

## 8. Merge Chunk Results into a Knowledge Graph

In [ ]:
canonical_map = build_character_canonical_map(chunk_results)
nodes_df = build_nodes_table(chunk_results, canonical_map)
edges_df = merge_relationships(chunk_results, canonical_map)

nodes_path = OUTPUT_DIR / f"{SELECTED_MOVIE_ID}_nodes.csv"
edges_path = OUTPUT_DIR / f"{SELECTED_MOVIE_ID}_edges.csv"
nodes_json_path = OUTPUT_DIR / f"{SELECTED_MOVIE_ID}_nodes.json"
edges_json_path = OUTPUT_DIR / f"{SELECTED_MOVIE_ID}_edges.json"

nodes_df.to_csv(nodes_path, index=False)
edges_df.to_csv(edges_path, index=False)
save_json(nodes_json_path, nodes_df.to_dict(orient="records"))
save_json(edges_json_path, edges_df.to_dict(orient="records"))

print("Saved:", nodes_path, edges_path)
edges_df.head(20)

## 9. Translate Dialogue with Relationship Context

For each line, the translation prompt includes the speaker's known relationships from the knowledge graph. This helps Gemini choose more natural tone, pronouns, intimacy, hostility, and politeness in Traditional Chinese.

In [ ]:
relationship_context = relationship_context_by_speaker(edges_df)
translation_turns = turns[:MAX_TRANSLATION_TURNS] if MAX_TRANSLATION_TURNS is not None else turns
translation_results = []

if RUN_TRANSLATION:
    if client is None:
        client = GeminiJsonClient(api_key="", model=GEMINI_MODEL)

for batch_index, batch in enumerate(batched(translation_turns, TRANSLATION_BATCH_SIZE), start=1):
    print(f"Translation batch {batch_index}: {len(batch)} turns")
    if RUN_TRANSLATION:
        result = translate_turn_batch_with_gemini(
            SELECTED_MOVIE_ID,
            SELECTED_MOVIE_TITLE,
            batch,
            relationship_context,
            client=client,
        )
    else:
        result = mock_translate_turns(
            SELECTED_MOVIE_ID,
            SELECTED_MOVIE_TITLE,
            batch,
            relationship_context,
        )
    payload = result.model_dump()
    save_json(TRANSLATION_OUTPUT_DIR / f"{SELECTED_MOVIE_ID}_translation_batch_{batch_index:04d}.json", payload)
    translation_results.extend(payload["translations"])

translations_df = pd.DataFrame(translation_results)
translation_csv_path = OUTPUT_DIR / f"{SELECTED_MOVIE_ID}_translations_zh_tw.csv"
translation_json_path = OUTPUT_DIR / f"{SELECTED_MOVIE_ID}_translations_zh_tw.json"
translations_df.to_csv(translation_csv_path, index=False)
save_json(translation_json_path, translation_results)

translations_df.head(20)

## 10. Optional Visualization

In [ ]:
try:
    import matplotlib.pyplot as plt
    import networkx as nx

    graph = nx.DiGraph()
    for _, row in nodes_df.iterrows():
        graph.add_node(row["name"])
    for _, row in edges_df.iterrows():
        graph.add_edge(row["source"], row["target"], label=row["relation_type"])

    if graph.number_of_nodes() > 0:
        plt.figure(figsize=(12, 9))
        pos = nx.spring_layout(graph, seed=42)
        nx.draw(graph, pos, with_labels=True, node_size=1200, font_size=8, arrows=True)
        edge_labels = {(u, v): data["label"] for u, v, data in graph.edges(data=True)}
        nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=7)
        plt.axis("off")
        plt.show()
except Exception as exc:
    print("Visualization skipped:", exc)

## 11. Full Command-Line Run

Use this command when you are ready to process the full movie with Gemini extraction and translation:

```powershell
python movie_kg_pipeline.py --movie-title "Titanic" --run-llm --translate --translation-batch-size 20
```

Start with `--max-chunks` and `--max-translation-turns` while testing.